In [1]:
import time, json, logging, re
from pathlib import Path
from datetime import datetime

SCRIPTS = Path('..') / 'ejemplos_codigo'
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
print('[OK] Entorno B11 listo')

[OK] Entorno B11 listo


# Orquestación, Agentes Colaborativos y Protocolo MCP

Este bloque es donde la IA deja de ser una herramienta aislada  
y se convierte en una pieza integrada en la arquitectura de software de la empresa.

```
Tool Calling  -->  Pipelines  -->  MCP  -->  Multi-Agente
   (conectar)     (orquestar)  (estandarizar)  (colaborar)
```

**Actividad de calentamiento - El proceso manual:**  
Se describe el proceso de alta de un nuevo cliente paso a paso.  
Los participantes identifican que pasos se pueden automatizar con IA  
y cuales requieren intervención humana.  
Esta es la mentalidad de orquestación.

> **Antes de seguir:** ¿qué proceso de tu empresa tiene más de tres pasos secuenciales donde cada paso depende del anterior, y cuál de esos pasos sería el más difícil de automatizar?

<details>
<summary>Orientación para el instructor (desplegar tras la reflexión)</summary>

**Una respuesta madura menciona al menos uno de estos elementos:**
- Un proceso concreto con pasos claros: alta de cliente, ciclo de venta, gestión de una incidencia crítica, proceso de onboarding
- La identificación del paso "cuello de botella" que requiere juicio humano, acceso a sistemas externos o aprobación explícita
- La intuición de que la automatización no es "todo o nada": se puede automatizar los pasos mecánicos y dejar los que requieren criterio humano como checkpoints

**Si nadie responde, preguntar:**
"¿Qué proceso de la empresa involucra a más de dos personas o sistemas diferentes para completarse? ¿Qué pasa cuando uno de esos pasos falla?"

**Señal de comprensión:**
El alumno puede describir un proceso con sus pasos y señalar cuál no se puede automatizar y por qué. Si además propone dónde pondría el checkpoint humano, ha entendido el diseño de orquestación antes de ver el código.

</details>

## 11.1 Tool Calling: Conectando el Modelo con el Mundo Real

Un LLM por si solo solo puede generar texto.  
No puede consultar una base de datos, enviar un email, crear un ticket en Jira,  
o verificar el estado de un pedido.

**Tool calling** es el mecanismo que permite al modelo invocar funciones externas  
como parte de su razonamiento.

```
Usuario: 'Cual es el stock del producto REF-4521?'
LLM razona: 'Necesito consultar el inventario'
LLM genera: {tool: 'consultar_stock', params: {product_id: 'REF-4521'}}
Sistema ejecuta: consultar_stock('REF-4521')  -->  {stock: 142, almacen: 'Madrid'}
LLM sintetiza: 'El producto REF-4521 tiene 142 unidades en Madrid-Centro.'
```

**El LLM no ejecuta la función directamente.**  
Genera una petición estructurada (JSON).  
La aplicación ejecuta la función real y devuelve el resultado al modelo.  

**La calidad de la descripción de cada herramienta es crítica.**  
El modelo decide que herramienta usar basándose en la descripción en lenguaje natural.

In [2]:
# Simulacion de un ciclo de tool calling completo
# Muestra el flujo sin llamadas reales a la API

# Definicion de herramientas (equivalente a lo que se envia a la API de Claude)
TOOLS = [
    {
        'name': 'consultar_stock',
        'description': (
            'Consulta el stock actual de un producto en el sistema de inventarios de la empresa. '
            'Usar cuando el usuario pregunte por disponibilidad o unidades de un producto.'
        ),
        'input_schema': {
            'type': 'object',
            'properties': {
                'product_id': {'type': 'string', 'description': 'Codigo de referencia (ej: REF-4521)'},
                'warehouse_id': {'type': 'string', 'description': 'ID del almacen. Opcional.'},
            },
            'required': ['product_id'],
        },
    },
    {
        'name': 'crear_ticket_soporte',
        'description': (
            'Crea un nuevo ticket en el sistema de soporte. Usar cuando el usuario reporta un '
            'problema que no puede resolverse en la conversacion.'
        ),
        'input_schema': {
            'type': 'object',
            'properties': {
                'titulo':       {'type': 'string'},
                'descripcion':  {'type': 'string'},
                'prioridad':    {'type': 'string', 'enum': ['critica', 'alta', 'media', 'baja']},
                'modulo':       {'type': 'string'},
            },
            'required': ['titulo', 'descripcion', 'prioridad'],
        },
    },
    {
        'name': 'listar_productos_bajo_stock',
        'description': (
            'Lista los productos cuyo stock esta por debajo del punto de pedido. '
            'Usar cuando se pidan alertas o reportes de stock critico.'
        ),
        'input_schema': {
            'type': 'object',
            'properties': {
                'warehouse_id': {'type': 'string'},
                'limit':        {'type': 'integer', 'description': 'Maximo de productos a retornar'},
            },
        },
    },
]

# Base de datos simulada
INVENTARIO_FAKE = {
    'REF-4521': {'stock': 142, 'almacen': 'Madrid-Centro', 'punto_pedido': 50},
    'REF-1023': {'stock': 8,   'almacen': 'Barcelona-Este', 'punto_pedido': 30},
    'REF-3301': {'stock': 3,   'almacen': 'Madrid-Centro', 'punto_pedido': 20},
    'REF-7781': {'stock': 245, 'almacen': 'Valencia-Norte', 'punto_pedido': 100},
}

def ejecutar_tool(tool_name: str, params: dict) -> dict:
    """Ejecutor de herramientas (capa de aplicacion, no el LLM)."""
    if tool_name == 'consultar_stock':
        pid = params['product_id']
        if pid not in INVENTARIO_FAKE:
            return {'error': f'Producto {pid} no encontrado'}
        return INVENTARIO_FAKE[pid]

    if tool_name == 'listar_productos_bajo_stock':
        limit = params.get('limit', 10)
        bajos = [
            {'sku': sku, **datos}
            for sku, datos in INVENTARIO_FAKE.items()
            if datos['stock'] < datos['punto_pedido']
        ]
        return {'productos': bajos[:limit], 'total': len(bajos)}

    if tool_name == 'crear_ticket_soporte':
        ticket_id = f'TKT-{int(time.time()) % 10000:04d}'
        return {'ticket_id': ticket_id, 'estado': 'creado', **params}

    return {'error': f'Herramienta desconocida: {tool_name}'}


# Simulacion del ciclo conversacional multi-tool
def simular_ciclo_tool_calling(consulta_usuario: str):
    """
    Simula el ciclo de tool calling sin llamada real al LLM.
    El 'LLM simulado' decide la herramienta por palabras clave.
    """
    print(f'Usuario: "{consulta_usuario}"')
    print()

    # El LLM (simulado) selecciona herramienta y parametros
    c = consulta_usuario.lower()
    if 'bajo stock' in c or 'critico' in c or 'alertas' in c:
        tool, params = 'listar_productos_bajo_stock', {'limit': 3}
    elif 'ref-' in c:
        ref = re.search(r'ref-\d+', c, re.IGNORECASE)
        tool, params = 'consultar_stock', {'product_id': ref.group(0).upper() if ref else 'REF-0000'}
    else:
        print('LLM: responde directamente sin herramienta')
        return

    print(f'LLM decide usar: {tool}({params})')
    resultado = ejecutar_tool(tool, params)
    print(f'Sistema ejecuta -> resultado: {json.dumps(resultado, ensure_ascii=False)}')
    print()
    print('LLM sintetiza la respuesta con el resultado:')

    if tool == 'consultar_stock':
        pid = params['product_id']
        if 'error' in resultado:
            print(f'  "No he encontrado el producto {pid} en el sistema."')
        else:
            print(f'  "El producto {pid} tiene {resultado["stock"]} unidades en {resultado["almacen"]}."')

    elif tool == 'listar_productos_bajo_stock':
        n = resultado['total']
        productos = ', '.join(p['sku'] for p in resultado['productos'])
        print(f'  "Hay {n} productos por debajo del punto de pedido: {productos}."')


print('=== Ciclo 1: consulta de stock especifico ===')
simular_ciclo_tool_calling('Cual es el stock del producto REF-4521?')
print()
print('=== Ciclo 2: reporte de stock critico ===')
simular_ciclo_tool_calling('Dame los productos con stock bajo o critico ahora mismo')

=== Ciclo 1: consulta de stock especifico ===
Usuario: "Cual es el stock del producto REF-4521?"

LLM decide usar: consultar_stock({'product_id': 'REF-4521'})
Sistema ejecuta -> resultado: {"stock": 142, "almacen": "Madrid-Centro", "punto_pedido": 50}

LLM sintetiza la respuesta con el resultado:
  "El producto REF-4521 tiene 142 unidades en Madrid-Centro."

=== Ciclo 2: reporte de stock critico ===
Usuario: "Dame los productos con stock bajo o critico ahora mismo"

LLM decide usar: listar_productos_bajo_stock({'limit': 3})
Sistema ejecuta -> resultado: {"productos": [{"sku": "REF-1023", "stock": 8, "almacen": "Barcelona-Este", "punto_pedido": 30}, {"sku": "REF-3301", "stock": 3, "almacen": "Madrid-Centro", "punto_pedido": 20}], "total": 2}

LLM sintetiza la respuesta con el resultado:
  "Hay 2 productos por debajo del punto de pedido: REF-1023, REF-3301."


## 11.2 Orquestación de Flujos Complejos

**Patrones de pipeline:**

```
Secuencial:  A --> B --> C --> D

Bifurcación: A --> clasificar --> [B si consulta | C si incidencia | D si solicitud]

Agregación:  A --> [B en paralelo] --> fusión --> D
                   [C en paralelo] /

Reintentos:  A --> B (falla) --> espera --> B (reintento) --> C
```

**Logging y Observabilidad** - las 4 dimensiones a monitorizar:
- **Técnica**: latencia por paso, coste, tasa de errores, tokens
- **Calidad**: precisión del modelo, tasa de fallbacks humanos
- **Negocio**: adopción, tiempo ahorrado, satisfacción
- **Alertas**: degradación de precisión, aumento de coste, latencia > SLA

In [3]:
# Pipeline con reintentos, logging y trazabilidad

import time, uuid

class PipelineStep:
    """Paso de un pipeline con reintentos y logging de traza."""

    def __init__(self, name: str, function, max_retries: int = 3,
                 fallo_simulado: bool = False):
        self.name = name
        self.function = function
        self.max_retries = max_retries
        self.fallo_simulado = fallo_simulado  # para demo

    def execute(self, input_data: dict, trace: list) -> dict:
        for attempt in range(self.max_retries):
            t0 = time.time()
            try:
                # Simular fallo en el primer intento
                if self.fallo_simulado and attempt == 0:
                    raise ConnectionError('Timeout de API (simulado)')
                result = self.function(input_data)
                elapsed_ms = int((time.time() - t0) * 1000)
                trace.append({
                    'step': self.name, 'status': 'success',
                    'attempt': attempt + 1, 'elapsed_ms': elapsed_ms,
                })
                return result
            except Exception as e:
                elapsed_ms = int((time.time() - t0) * 1000)
                trace.append({
                    'step': self.name, 'status': 'retry',
                    'attempt': attempt + 1, 'error': str(e), 'elapsed_ms': elapsed_ms,
                })
                if attempt < self.max_retries - 1:
                    espera = 2 ** attempt * 0.1  # backoff exponencial reducido para demo
                    time.sleep(espera)
                else:
                    raise


class Pipeline:
    """Pipeline secuencial con trazabilidad end-to-end."""

    def __init__(self, steps: list):
        self.steps = steps

    def run(self, input_data: dict) -> tuple:
        trace_id = str(uuid.uuid4())[:8]
        trace = []
        t_total = time.time()
        current = input_data
        for step in self.steps:
            current = step.execute(current, trace)
        elapsed_total = int((time.time() - t_total) * 1000)
        return current, {'trace_id': trace_id, 'total_ms': elapsed_total, 'steps': trace}


# Funciones de los pasos (simulacion)
def clasificar_ticket(data: dict) -> dict:
    tiempo_proceso = 0.05  # 50ms simulados
    time.sleep(tiempo_proceso)
    desc = data.get('descripcion', '').lower()
    cat = 'bug' if 'error' in desc else 'feature' if 'gustaria' in desc else 'pregunta'
    return {**data, 'categoria': cat, 'confianza': 0.91}

def extraer_entidades(data: dict) -> dict:
    time.sleep(0.02)
    modulos_conocidos = ['inventarios', 'ventas', 'facturacion', 'compras']
    desc = data.get('descripcion', '').lower()
    modulo = next((m for m in modulos_conocidos if m in desc), 'general')
    return {**data, 'modulo_detectado': modulo}

def buscar_solucion_rag(data: dict) -> dict:
    time.sleep(0.04)
    # Simula una busqueda RAG
    soluciones = {
        'bug': 'Revisar logs del servidor. Issue conocido: limpiar cache de sesion.',
        'feature': 'Registrar como solicitud de mejora en el backlog.',
        'pregunta': 'Consultar documentacion en seccion FAQ del modulo.',
    }
    cat = data.get('categoria', 'pregunta')
    return {**data, 'solucion_sugerida': soluciones[cat]}

def generar_respuesta(data: dict) -> dict:
    time.sleep(0.02)
    return {**data, 'respuesta_generada': (
        f"Hemos clasificado tu consulta como [{data['categoria']}] "
        f"en el modulo {data['modulo_detectado']}. "
        f"Sugerencia: {data['solucion_sugerida']}"
    )}


# Construir y ejecutar el pipeline
pipeline = Pipeline(steps=[
    PipelineStep('clasificar',      clasificar_ticket,   fallo_simulado=True),  # falla 1 vez
    PipelineStep('extraer_ent',     extraer_entidades),
    PipelineStep('buscar_rag',      buscar_solucion_rag),
    PipelineStep('generar_resp',    generar_respuesta),
])

ticket = {
    'id': 'TKT-2541',
    'descripcion': 'Al intentar exportar el informe de inventarios sale un error 500',
}

resultado, traza = pipeline.run(ticket)

print('=== Traza del pipeline ===')
print(f'Trace ID: {traza["trace_id"]} | Total: {traza["total_ms"]}ms')
print()
for s in traza['steps']:
    icono = '[OK]  ' if s['status'] == 'success' else '[!] '
    extra = f" | retry: {s['error'][:40]}" if 'error' in s else ''
    print(f"  {icono} {s['step']:<16} intento {s['attempt']} | {s['elapsed_ms']}ms{extra}")

print()
print('=== Respuesta generada ===')
print(f'  {resultado["respuesta_generada"]}')

=== Traza del pipeline ===
Trace ID: 49d906f8 | Total: 233ms

  [!]  clasificar       intento 1 | 0ms | retry: Timeout de API (simulado)
  [OK]   clasificar       intento 2 | 50ms
  [OK]   extraer_ent      intento 1 | 20ms
  [OK]   buscar_rag       intento 1 | 40ms
  [OK]   generar_resp     intento 1 | 20ms

=== Respuesta generada ===
  Hemos clasificado tu consulta como [bug] en el modulo inventarios. Sugerencia: Revisar logs del servidor. Issue conocido: limpiar cache de sesion.


## 11.3 Model Context Protocol (MCP)

MCP es un protocolo abierto (propuesto por Anthropic) que estandariza  
la forma en que los modelos se conectan con herramientas, bases de datos y servicios.

**Analogía para un desarrollador .NET:**  
MCP es al tool calling lo que una interfaz (IService) es a una implementación concreta.  
Define un contrato que desacopla al consumidor (el modelo) del proveedor (la herramienta).

```
[LLM / Cliente MCP]
         |
         | (protocolo estandarizado JSON-RPC)
         |
    +----+----+----+
    |         |         |
[Servidor  [Servidor  [Servidor
 MCP:       MCP:       MCP:
 Base de    Sistema    API de
 Datos]     Archivos]  la empresa]
```

Un servidor MCP expone tres tipos de capacidades:
1. **Tools**: funciones invocables
2. **Resources**: datos que el modelo puede leer
3. **Prompts**: plantillas predefinidas que el servidor ofrece

**Ventajas de MCP para la empresa:**
- Reutilización: un servidor MCP se escribe una vez y funciona con cualquier cliente MCP
- Seguridad: el servidor controla que operaciones expone
- Independencia del modelo: si se cambia de Claude a GPT, las integraciones siguen funcionando
- Composabilidad: multiples servidores MCP combinados

In [4]:
# Definicion conceptual de un servidor MCP para Sistema de Inventarios
# En produccion esto seria un servidor Python con el SDK de MCP

SERVIDOR_MCP_EMPRESA = {
    'name': 'empresa-inventarios',
    'version': '1.0.0',
    'description': 'Servidor MCP para el modulo de inventarios de la empresa',

    # Herramientas invocables
    'tools': {
        'consultar_stock': {
            'description': 'Consulta stock actual de un producto',
            'params': {'product_id': 'string', 'warehouse_id': 'string?'},
        },
        'registrar_movimiento': {
            'description': 'Registra una entrada o salida de inventario',
            'params': {
                'product_id': 'string', 'tipo': 'entrada|salida',
                'cantidad': 'integer', 'motivo': 'string',
            },
        },
        'generar_informe_stock': {
            'description': 'Genera un informe de stock por almacen',
            'params': {'warehouse_id': 'string', 'formato': 'json|csv'},
        },
    },

    # Recursos de datos accesibles
    'resources': {
        'catalogo_productos': 'Lista completa de productos con sus atributos',
        'almacenes': 'Lista de almacenes con ubicaciones y capacidades',
        'movimientos_recientes': 'Ultimos 100 movimientos de inventario',
    },

    # Plantillas de prompts predefinidas
    'prompts': {
        'analisis_stock_bajo': (
            'Analiza los productos con stock por debajo del punto de pedido '
            'y sugiere acciones prioritarias.'
        ),
        'reconciliacion': (
            'Compara el stock fisico reportado con el stock del sistema '
            'e identifica discrepancias con posibles causas.'
        ),
    },
}


# Comparativa: sin MCP vs con MCP
print('=== Sin MCP: cada integracion es codigo custom ===')
print()
codigo_sin_mcp = '''
# Para Claude Code:
claude_tools = [{'name': 'consultar_stock', ...}]

# Para el chatbot interno (otra implementacion):
chatbot_tools = [{'function_name': 'get_stock', ...}]  # formato diferente

# Para el agente de soporte (otra implementacion):
soporte_tools = [{'action': 'query_inventory', ...}]   # formato diferente

# -> 3 implementaciones distintas del mismo endpoint de inventarios
'''
print(codigo_sin_mcp)

print('=== Con MCP: un servidor, multiples clientes ===')
codigo_con_mcp = '''
# Un servidor MCP de inventarios:
# empresa-inventarios v1.0.0

# Claude Code lo consume -> OK
# Chatbot interno lo consume -> OK (mismo protocolo)
# Agente de soporte lo consume -> OK (mismo protocolo)
# Manana se cambia de Claude a GPT -> OK (MCP es agnóstico del modelo)

# -> 1 implementacion, N clientes
'''
print(codigo_con_mcp)

print('=== Inventario del servidor MCP ===')
print(f'Nombre: {SERVIDOR_MCP_EMPRESA["name"]} v{SERVIDOR_MCP_EMPRESA["version"]}')
print(f'Tools:     {list(SERVIDOR_MCP_EMPRESA["tools"].keys())}')
print(f'Resources: {list(SERVIDOR_MCP_EMPRESA["resources"].keys())}')
print(f'Prompts:   {list(SERVIDOR_MCP_EMPRESA["prompts"].keys())}')

=== Sin MCP: cada integracion es codigo custom ===


# Para Claude Code:
claude_tools = [{'name': 'consultar_stock', ...}]

# Para el chatbot interno (otra implementacion):
chatbot_tools = [{'function_name': 'get_stock', ...}]  # formato diferente

# Para el agente de soporte (otra implementacion):
soporte_tools = [{'action': 'query_inventory', ...}]   # formato diferente

# -> 3 implementaciones distintas del mismo endpoint de inventarios

=== Con MCP: un servidor, multiples clientes ===

# Un servidor MCP de inventarios:
# empresa-inventarios v1.0.0

# Claude Code lo consume -> OK
# Chatbot interno lo consume -> OK (mismo protocolo)
# Agente de soporte lo consume -> OK (mismo protocolo)
# Manana se cambia de Claude a GPT -> OK (MCP es agnóstico del modelo)

# -> 1 implementacion, N clientes

=== Inventario del servidor MCP ===
Nombre: empresa-inventarios v1.0.0
Tools:     ['consultar_stock', 'registrar_movimiento', 'generar_informe_stock']
Resources: ['catalogo_productos', 'almacenes

## 11.4 Sistemas Multi-Agente

**Por que multiples agentes:**  
Un único agente con todas las herramientas se vuelve frágil para tareas complejas.  
Cada agente se especializa en un dominio y colabora con los demas.

**Patrón Planner-Executor:**
```
[USUARIO] --> [PLANNER: descompone la tarea en pasos]
                    |
              [EXECUTOR: coordina la ejecución]
                    |
         +----------+-----------+
         |          |           |
    [Agente    [Agente    [Agente
     Inventario] Compras]  Notif.]
```

**Antipatrones a evitar:**
- Over-engineering: si un prompt con tools lo resuelve, no usar multi-agente
- Loops infinitos: siempre definir máximo de iteraciones y condición de terminación
- Pérdida de contexto: el estado compartido debe estar bien definido
- Coste descontrolado: 5 agentes x 3 llamadas = 15 llamadas al LLM por petición
- Sin observabilidad: diagnosticar por que un sistema multi-agente falló es imposible sin trazas

**Regla práctica:**  
Empezar con un único agente con tools.  
Escalar a multi-agente solo cuando la complejidad del dominio lo exija  
y se tenga infraestructura de observabilidad para gestionarlo.

In [5]:
# Simulacion simplificada del patron Planner-Executor

class Agente:
    """Agente especializado con acceso a un subconjunto de herramientas."""

    def __init__(self, nombre: str, capacidades: list):
        self.nombre = nombre
        self.capacidades = capacidades

    def puede_ejecutar(self, tarea: str) -> bool:
        return any(cap in tarea.lower() for cap in self.capacidades)

    def ejecutar(self, tarea: str, contexto: dict) -> dict:
        time.sleep(0.02)  # simula trabajo real
        resultado = f'[{self.nombre}] ejecuto: {tarea}'
        print(f'    -> {resultado}')
        return {**contexto, f'resultado_{self.nombre}': resultado}


class Planner:
    """Agente coordinador que descompone tareas en pasos."""

    def descomponer(self, tarea_usuario: str) -> list:
        """Descompone la tarea del usuario en pasos ejecutables."""
        # En produccion esto seria una llamada al LLM
        planes = {
            'reponer': [
                'verificar stock bajo minimo',
                'consultar proveedor disponible',
                'generar orden de compra',
                'notificar al responsable',
            ],
            'rfp': [
                'extraer requisitos del documento',
                'evaluar cobertura de la empresa',
                'generar borrador de respuesta',
                'revisar calidad del borrador',
            ],
        }
        for keyword, plan in planes.items():
            if keyword in tarea_usuario.lower():
                return plan
        return ['analizar la tarea', 'ejecutar accion', 'reportar resultado']


class Executor:
    """Coordina la ejecucion del plan asignando cada paso al agente correcto."""

    def __init__(self, agentes: list, max_iteraciones: int = 10):
        self.agentes = agentes
        self.max_iter = max_iteraciones

    def ejecutar_plan(self, pasos: list, contexto: dict) -> dict:
        for i, paso in enumerate(pasos[:self.max_iter]):
            agente = next((a for a in self.agentes if a.puede_ejecutar(paso)), None)
            if agente:
                print(f'  Paso {i+1}/{len(pasos)}: "{paso}" -> asignado a {agente.nombre}')
                contexto = agente.ejecutar(paso, contexto)
            else:
                print(f'  Paso {i+1}/{len(pasos)}: "{paso}" -> sin agente disponible [!]')
        return contexto


# Construir el sistema multi-agente
agentes = [
    Agente('AgenteInventario',   ['stock', 'inventario', 'reponer', 'verificar']),
    Agente('AgenteCompras',      ['proveedor', 'orden', 'compra']),
    Agente('AgenteNotificacion', ['notificar', 'alerta', 'email']),
    Agente('AgenteDocumentos',   ['extraer', 'rfp', 'requisito', 'documento']),
    Agente('AgenteLLM',          ['generar', 'analizar', 'revisar', 'evaluar', 'borrador']),
]

planner = Planner()
executor = Executor(agentes, max_iteraciones=10)

# Ejecutar tarea compleja
tarea = 'Reponer inventario critico del almacen Madrid-Centro'
print(f'=== Sistema Multi-Agente ===')
print(f'Tarea del usuario: "{tarea}"')
print()
plan = planner.descomponer(tarea)
print(f'Plan descompuesto por el Planner: {plan}')
print()
print('Executor asigna cada paso:')
resultado_final = executor.ejecutar_plan(plan, {'tarea': tarea})
print()
print(f'Pasos completados: {len([k for k in resultado_final if k.startswith("resultado")])}/{len(plan)}')

=== Sistema Multi-Agente ===
Tarea del usuario: "Reponer inventario critico del almacen Madrid-Centro"

Plan descompuesto por el Planner: ['verificar stock bajo minimo', 'consultar proveedor disponible', 'generar orden de compra', 'notificar al responsable']

Executor asigna cada paso:
  Paso 1/4: "verificar stock bajo minimo" -> asignado a AgenteInventario
    -> [AgenteInventario] ejecuto: verificar stock bajo minimo
  Paso 2/4: "consultar proveedor disponible" -> asignado a AgenteCompras
    -> [AgenteCompras] ejecuto: consultar proveedor disponible
  Paso 3/4: "generar orden de compra" -> asignado a AgenteCompras
    -> [AgenteCompras] ejecuto: generar orden de compra
  Paso 4/4: "notificar al responsable" -> asignado a AgenteNotificacion
    -> [AgenteNotificacion] ejecuto: notificar al responsable

Pasos completados: 3/4


---
## 7. Ejercicio de Decisión: ¿usarias IA aquí?

### Caso: sistema multi-agente para el ciclo de venta de la empresa

El equipo de ventas quiere un sistema de 4 agentes coordinados que gestione
el ciclo completo de una oportunidad: (1) analizar la RFP, (2) identificar módulos
relevantes de la empresa, (3) generar propuesta técnica, (4) calcular precio y
enviar borrador al comercial. Todo sin intervención humana entre pasos.

---

**Pregunta 1 - El riesgo principal**
¿Cual es el mayor riesgo de concatenar 4 agentes de forma completamente automática?
¿Que ocurre si el agente 1 comete un error?

**Pregunta 2 - Los checkpoints obligatorios**
¿En que punto del flujo pondrias un checkpoint humano obligatorio?
¿Por que ese punto y no otro?

**Pregunta 3 - La detección de errores de precio**
Si el agente de cálculo de precios comete un error, ¿como lo detectarias
antes de que el borrador llegue al cliente? ¿Que validación automática implementarias?

**Pregunta 4 - El enfoque de implementación**
¿Implementarias los 4 agentes desde el principio o lo harias por fases?
Describe las fases y justifica el orden.

---
*Escribe tus respuestas en la celda siguiente.*

### Mis respuestas

**Pregunta 1 - El riesgo principal:**

*(escribe aquí)*

**Pregunta 2 - Los checkpoints obligatorios:**

*(escribe aquí)*

**Pregunta 3 - La detección de errores de precio:**

*(escribe aquí)*

**Pregunta 4 - El enfoque de implementación:**

*(escribe aquí)*

---

<!--
CRITERIOS DE Evaluación (para el instructor)

Pregunta 1 - El riesgo principal:
Respuesta correcta: propagacion de errores en cascada.
Si el agente 1 (análisis de RFP) malinterpreta el alcance del proyecto,
los agentes 2, 3 y 4 construyen sobre esa base erronea y el borrador final
puede ser completamente incorrecto en propuesta técnica y precio.
En un sistema completamente automático, este error llega al comercial
sin ninguna oportunidad de correccion intermedia.
Insuficiente: "puede cometer errores" sin explicar el mecanismo de propagacion.

Pregunta 2 - Los checkpoints obligatorios:
El checkpoint mas crítico: entre el agente 1 (análisis de RFP) y el agente 2
(identificacion de módulos), porque es donde se define el alcance.
Un error aquí invalida todo lo que sigue.
Segundo checkpoint valido: antes de que el borrador salga del sistema,
el comercial debe aprobarlo antes de enviarlo al cliente.
Insuficiente: poner checkpoint "en cada paso" sin justificar por que ese y no otro.

Pregunta 3 - La detección de errores de precio:
Validaciones automáticas validas:
 - Comparar el precio calculado con el rango histórico de propuestas similares (por tamanio de cliente, módulos)
 - Verificar que la suma de componentes coincide con el total
 - Alertar si el precio supera o es inferior al umbral definido para el tipo de cliente
 - Requerir autorización si el descuento supera un porcentaje máximo
Insuficiente: "el comercial lo revisa" sin especificar que valida automaticamente el sistema.

Pregunta 4 - El enfoque de implementación:
Fases correctas:
  Fase 1: automatizar solo el agente 1 (análisis de RFP) con revisión humana antes de continuar
  Fase 2: añadir agente 2 (módulos relevantes) y validar la selección antes de generar propuesta
  Fase 3: generar propuesta técnica con revisión humana
  Fase 4: calcular precio con validación automática + aprobacion humana
  Fase 5 (opcional): automatizar el flujo completo con métricas de calidad establecidas
Incorrecto: implementar los 4 agentes desde el principio sin validación por fases.
-->


## Resumen del Bloque 11

```
Tool Calling  =  LLM + herramientas externas
Pipeline      =  tool calling + lógica de negocio + manejo de errores
MCP           =  protocolo estándar para desacoplar agentes de herramientas
Multi-Agente  =  colaboración entre agentes especializados
```

**Cuando escalar:**

| Escenario | Solución recomendada |
|---|---|
| Consulta puntual + 1-2 herramientas | Tool calling simple |
| Flujo lineal de 3-6 pasos | Pipeline secuencial |
| Multiples tipos de entrada | Pipeline con bifurcación |
| Varias APIs en producción | MCP + servidor dedicado |
| Tarea compleja multi-dominio | Multi-agente Planner-Executor |

**Siguiente bloque (B12):** IA Aplicada al Negocio - 
los 8 casos de uso concretos de la empresa con su variable objetivo definida.